# NODEX · Revisión conjunta R1, R2 y R3

Carga de datos originales para análisis posterior, sin inferencia ni cálculo de métricas. En Colab, suba los tres ZIP al panel Archivos. Este notebook puede descomprimir cada uno en una carpeta con su identificador si la carpeta aún no existe. También funciona con las carpetas ya extraídas. Las claves son case_id, repeticion y run_id.


In [ ]:
from pathlib import Path
import json, hashlib, zipfile
try:
    from IPython.display import display
except ImportError:
    display = print
VERSIONS = {1: 'oficial-12casos-r1-20261028', 2: 'oficial-12casos-r2-20261103', 3: 'oficial-12casos-r3-20261103'}
BASE = Path.cwd()
def locate(version):
    for directory in [BASE / version, BASE / 'extracted_data' / version]:
        if (directory / 'extraction.json').is_file():
            return directory
    for archive in [BASE / (version + '.zip'), BASE / 'extracted_data' / (version + '.zip')]:
        if archive.is_file():
            directory = archive.parent / version
            if directory.exists():
                raise ValueError('Carpeta existente incompleta: ' + str(directory))
            with zipfile.ZipFile(archive) as z:
                for item in z.infolist():
                    if not (directory / item.filename).resolve().is_relative_to(directory.resolve()):
                        raise ValueError('Ruta ZIP inválida')
                z.extractall(directory)
            return directory
    raise FileNotFoundError('Suba el ZIP o ajuste BASE para ' + version)
FOLDERS = {rep: locate(version) for rep, version in VERSIONS.items()}
def load(folder, name):
    path = folder / name
    return [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines()] if path.exists() else []
def table(rows):
    try:
        import pandas as pd
    except ImportError:
        display(rows)
    else:
        display(pd.DataFrame(rows))


## Identidades e integridad
Estos controles comprueban archivos e identidades; no califican predicciones.


In [ ]:
run_index = [row for rep, folder in FOLDERS.items() for row in load(folder, 'run_status.jsonl')]
assert len(run_index) == 36
assert len({row['run_id'] for row in run_index}) == 36
assert len({(row['case_id'], row['repeticion']) for row in run_index}) == 36
for rep, folder in FOLDERS.items():
    for item in json.loads((folder / 'manifest.json').read_text(encoding='utf-8')):
        if item['available']:
            assert hashlib.sha256((folder / item['export_path']).read_bytes()).hexdigest() == item['sha256_copy'] == item['sha256_source']
table(run_index)


## Carga de los cinco escenarios
Los registros permanecen sin transformar. Referencias, predicciones y candidatos RAG se mantienen en tablas separadas.


In [ ]:
FILES = ['detection/predictions.jsonl', 'detection/verdicts.jsonl', 'detection/ground_truth.jsonl', 'reconstruction/predictions.jsonl', 'reconstruction/predictions_items.jsonl', 'reconstruction/reference.jsonl', 'mitre/attributions.jsonl', 'mitre/attributions_items.jsonl', 'mitre/ground_truth.jsonl', 'mitre/retrieved_candidates.jsonl', 'reports/text.jsonl', 'reports/sections.jsonl', 'performance/runs.jsonl', 'performance/llm_calls_items.jsonl', 'performance/processes.jsonl']
tables = {name: [row for rep, folder in FOLDERS.items() for row in load(folder, name)] for name in FILES}
for name in FILES:
    print(name, '· registros cargados:', len(tables[name]))


## Inspección del mismo caso en las tres repeticiones
Cambie case_id. event_id y evidence_uids conservan sus significados originales.


In [ ]:
case_id = 'uwf_eva_visible_0004'
for name in ['detection/predictions.jsonl', 'reconstruction/predictions_items.jsonl', 'mitre/attributions_items.jsonl', 'reports/text.jsonl']:
    print(name)
    table([row for row in tables[name] if row['case_id'] == case_id])


## Medidas de rendimiento registradas
Una fila por corrida; no se calculan promedios, métricas ni diferencias.


In [ ]:
performance_records = [dict(case_id=row['case_id'], repeticion=row['repeticion'], run_id=row['run_id'], **row['value']) for row in tables['performance/runs.jsonl'] if row['present'] and isinstance(row['value'], dict)]
table(performance_records)
